# Experiment 8 — Random Forest × 5 Imbalance Techniques

Random Forest builds many trees in parallel on random subsets (bagging).
Different from boosting — less prone to overfitting, more robust to noise.

**RF imbalance support:**
- Class weights: `class_weight='balanced'` or manual weights
- SMOTE: same as other models
- Focal Loss: NOT natively supported — uses threshold optimization instead
- Threshold: standard scan

**Note:** RF is slower than XGBoost/LightGBM at n_estimators=500 on large data.

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Equivalent to XGBoost params for fair comparison
RF_PARAMS  = dict(n_estimators=500, max_depth=6, random_state=42, n_jobs=-1)
SMOTE_CAP  = 500_000
THRESHOLDS = np.arange(0.1, 0.91, 0.01)
print("Experiment 8 — Random Forest × 5 Techniques")

Experiment 8 — Random Forest × 5 Techniques


In [2]:
def run_rf_technique(X_train, y_train, X_test, y_test, technique, v):
    t0 = time.time()

    if technique == 'baseline':
        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        return compute_all_metrics(y_test, probs, preds, time.time()-t0), probs

    elif technique == 'class_weights':
        model = RandomForestClassifier(**RF_PARAMS, class_weight='balanced')
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        return compute_all_metrics(y_test, probs, preds, time.time()-t0), probs

    elif technique == 'smote':
        if v == 'A' and len(X_train) > SMOTE_CAP:
            idx = np.random.RandomState(42).choice(len(X_train), SMOTE_CAP, replace=False)
            X_s, y_s = X_train[idx], y_train[idx]
        else:
            X_s, y_s = X_train, y_train
        X_res, y_res = SMOTE(random_state=42).fit_resample(X_s, y_s)
        smote_t = time.time() - t0
        model   = RandomForestClassifier(**RF_PARAMS)
        model.fit(X_res, y_res)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        return compute_all_metrics(y_test, probs, preds, time.time()-t0+smote_t), probs

    elif technique == 'focal_loss':
        # RF does not support custom objectives — use threshold opt instead
        # Noted as limitation: focal loss not applicable to sklearn RF
        print(f"    [NOTE] RF does not support focal loss — using threshold optimization instead.")
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2,
                                                      random_state=42, stratify=y_train)
        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X_tr, y_tr)
        val_probs = model.predict_proba(X_val)[:,1]
        f1s   = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
                 for t in THRESHOLDS]
        best_t = THRESHOLDS[int(np.argmax(f1s))]
        probs  = model.predict_proba(X_test)[:,1]
        preds  = (probs >= best_t).astype(int)
        m = compute_all_metrics(y_test, probs, preds, time.time()-t0)
        m['Best_Threshold'] = round(best_t, 2)
        return m, probs

    elif technique == 'threshold':
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2,
                                                      random_state=42, stratify=y_train)
        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X_tr, y_tr)
        val_probs = model.predict_proba(X_val)[:,1]
        f1s    = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
                  for t in THRESHOLDS]
        best_t = THRESHOLDS[int(np.argmax(f1s))]
        probs  = model.predict_proba(X_test)[:,1]
        preds  = (probs >= best_t).astype(int)
        m = compute_all_metrics(y_test, probs, preds, time.time()-t0)
        m['Best_Threshold'] = round(best_t, 2)
        return m, probs

In [3]:
techniques = ['baseline','class_weights','smote','focal_loss','threshold']
tech_names = {'baseline':'Baseline','class_weights':'Class Weights',
              'smote':'SMOTE','focal_loss':'Focal Loss','threshold':'Threshold Opt.'}
all_results = []

for v in ['A','B','C']:
    print(f"\n[Exp8-RF] Version {v}")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train = train.drop('label',axis=1).values; y_train = train['label'].values
    X_test  = test.drop('label',axis=1).values;  y_test  = test['label'].values

    for tech in techniques:
        print(f"  [{tech}] running...")
        metrics, probs = run_rf_technique(X_train, y_train, X_test, y_test, tech, v)
        metrics['Version']   = v
        metrics['Technique'] = tech_names[tech]
        all_results.append(metrics)
        np.save(os.path.join(RESULTS_DIR, f'probs_rf_{tech}_version_{v}.npy'), probs)
        print_metrics(metrics, label=f"RF-{tech} V{v}")

print("\n[Exp8-RF] All complete.")


[Exp8-RF] Version A
  [baseline] running...
[RF-baseline VA] AUC=0.7607 | F1=0.7196 | Signal_Sig=169.8559 | Train_Time=207.31s
  [class_weights] running...
[RF-class_weights VA] AUC=0.7609 | F1=0.6991 | Signal_Sig=170.3351 | Train_Time=182.76s
  [smote] running...
[RF-smote VA] AUC=0.7619 | F1=0.7004 | Signal_Sig=170.3080 | Train_Time=330.9s
  [focal_loss] running...
    [NOTE] RF does not support focal loss — using threshold optimization instead.
[RF-focal_loss VA] AUC=0.7606 | F1=0.7362 | Signal_Sig=169.8636 | Train_Time=156.32s
  [threshold] running...
[RF-threshold VA] AUC=0.7606 | F1=0.7362 | Signal_Sig=169.8636 | Train_Time=148.81s

[Exp8-RF] Version B
  [baseline] running...
[RF-baseline VB] AUC=0.7650 | F1=0.0000 | Signal_Sig=3.6380 | Train_Time=74.34s
  [class_weights] running...
[RF-class_weights VB] AUC=0.7659 | F1=0.2936 | Signal_Sig=21.5653 | Train_Time=73.63s
  [smote] running...
[RF-smote VB] AUC=0.7302 | F1=0.2815 | Signal_Sig=21.7944 | Train_Time=191.35s
  [focal_loss

In [4]:
cols = ['Version','Technique','AUC','F1','Signal_Significance','Train_Time_sec']
results_df = pd.DataFrame(all_results)
results_df = results_df[[c for c in cols if c in results_df.columns]]
save_path  = os.path.join(RESULTS_DIR, 'experiment8_random_forest.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      Technique      AUC       F1  Signal_Significance  Train_Time_sec
      A       Baseline 0.760652 0.719615           169.855927          207.31
      A  Class Weights 0.760947 0.699098           170.335093          182.76
      A          SMOTE 0.761865 0.700361           170.307986          330.90
      A     Focal Loss 0.760639 0.736209           169.863630          156.32
      A Threshold Opt. 0.760639 0.736209           169.863630          148.81
      B       Baseline 0.764979 0.000000             3.638034           74.34
      B  Class Weights 0.765933 0.293612            21.565256           73.63
      B          SMOTE 0.730191 0.281502            21.794402          191.35
      B     Focal Loss 0.764591 0.353408             3.837613           62.90
      B Threshold Opt. 0.764591 0.353408             3.837613           60.09
      C       Baseline 0.743665 0.000000             0.000000           61.40
      C  Class Weights 0.754799 0.084414             4.505818   